

# Graph Traversal: Reachability, Paths, and Components


```{contents}
:local:
:depth: 2
```


A facilities team needs to know which rooms can be reached, how many unweighted links a route needs, and which rooms form disconnected groups. A representation stores connections; traversal systematically discovers what those connections imply. Start with declared vertices so isolated rooms remain visible. Every example includes the complete model. The implementation uses ordinary classes, loops, a queue, and a visited set.


## Breadth-First Search and Discovery

```{index} Breadth-First Search and Discovery
```


Breadth-first search (BFS) discovers vertices in layers. Mark a vertex when it enters the queue, so two incoming edges cannot enqueue it twice. The source has distance zero. Each newly discovered neighbor receives its predecessor distance plus one. Queue order ensures that all vertices at distance d are processed before distance d+1. These are minimum edge counts for an unweighted graph; they are not minimum travel times when weights differ.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
foreach (string room in graph.Vertices())
    Console.WriteLine($"{room}: {result.DistanceTo(room)?.ToString() ?? "unreachable"}");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby: 0
Hall: 1
Lab: 2
Office: 2
Remote: unreachable


## Recover a Path with Predecessors

```{index} Recover a Path with Predecessors
```


Store a predecessor only at first discovery. To recover a route, follow predecessors from the destination back to the source and reverse the collected list. The source route contains the source itself; an unreachable target produces an empty array. Neighbor insertion order chooses among equally short routes. Test the distance and valid edges instead of assuming an arbitrary tie order is a mathematical requirement.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
Console.WriteLine(string.Join(" -> ", result.PathTo("Lab")));
Console.WriteLine($"Remote route length: {result.PathTo("Remote").Length}");
Console.WriteLine(string.Join(" -> ", result.PathTo("Lobby")));

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby -> Hall -> Lab
Remote route length: 0
Lobby


## Depth-First Search and Cycles

```{index} Depth-First Search and Cycles
```


Depth-first search (DFS) follows a branch before returning to another neighbor. Mark before following neighbors: a cycle then leads to an already visited vertex rather than infinite recursion. This teaching implementation uses recursive calls; the runtime call stack records unfinished work. It can require O(V) stack space and very deep graphs may exceed runtime stack limits. Part VII develops further traversal techniques. DFS discovers reachability, but its discovery depth does not promise a shortest route.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
Console.WriteLine(string.Join(", ", graph.Dfs("Lobby")));
Console.WriteLine(string.Join(", ", graph.Bfs("Lobby").VisitOrder()));

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby, Hall, Lab, Office
Lobby, Hall, Lab, Office


## Connected Components and Direction

```{index} Connected Components and Direction
```


For an undirected graph, start a traversal at each still-unvisited vertex. Each new start discovers one connected component. An isolated vertex forms its own component; an empty graph has none. A directed reachability result depends on the starting vertex, so it is not automatically an undirected connected component. The Components method rejects directed graphs rather than silently choosing a different definition.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
foreach (string[] component in graph.Components()) Console.WriteLine(string.Join(", ", component));
SimpleGraph directed = new SimpleGraph(true);
directed.AddVertex("Approve"); directed.AddVertex("Pay"); directed.AddEdge("Approve", "Pay");
Console.WriteLine(directed.Bfs("Pay").DistanceTo("Approve") is null);

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby, Hall, Lab, Office
Remote
True


## Costs and Contracts

```{index} Costs and Contracts
```


With adjacency lists and expected constant-time dictionary and set operations, a complete traversal takes O(V+E) time: each vertex is discovered once and each stored adjacency is examined once. An undirected edge is stored twice, which changes a constant factor. Visited state, predecessor state, and the queue or recursion stack require O(V) auxiliary space. Matrix traversal instead scans V potential neighbors per visited vertex and can take O(V²). Graph snapshots returned by Vertices and Neighbors protect the stored lists; the strings themselves are immutable. Unknown sources are rejected. The examples do not modify the graph during a search.


## Practice: Check a Reachability Report

```{index} Practice: Check a Reachability Report
```


Complete the assertion for the two-edge route to Lab, then add a check for a missing route to Remote.


In [ ]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
if (false /* TODO: detect the wrong Lab distance */ || result.DistanceTo("Remote") is not null) throw new Exception("Reachability failed.");
if (result.PathTo("Lobby").Length != 1) throw new Exception("Source path failed.");
Console.WriteLine("Layer and path checks passed.");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
if (result.DistanceTo("Lab") != 2 || result.DistanceTo("Remote") is not null) throw new Exception("Reachability failed.");
if (result.PathTo("Lobby").Length != 1) throw new Exception("Source path failed.");
Console.WriteLine("Layer and path checks passed.");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Layer and path checks passed.


```{rubric} Footnotes
```
[^1]: BFS minimizes the number of edges only when edges have equal cost. Weighted shortest paths belong to Part VII.
